# Sentiment Analysis using Amazon Review Dataset
## Model Development, Improvement and Evaluation

## 1. Introduction
This project develops a sentiment analysis model using the Amazon review dataset.
The objective is to classify reviews as positive or negative and improve model performance
through data enhancement and optimisation techniques.

## 2. Dataset Description  

The dataset used in this project is the Amazon Review Dataset, which contains customer reviews collected from multiple product categories. The dataset includes both positive and negative reviews, making it suitable for sentiment classification tasks.

Initially, only the **books domain** was used, consisting of 1000 positive and 1000 negative reviews. This provided a baseline for evaluating model performance.

To improve the model, additional domains were incorporated, including:
- Books  
- DVD  
- Electronics  
- Kitchen & Housewares  

Each domain contains labelled reviews (positive = 1, negative = 0). By combining all four domains, the dataset increased to approximately 8000 reviews, improving data diversity and enabling better model generalisation.

## 3. Data Preparation  

This section prepares the dataset for sentiment analysis.  
The process includes uploading files, extracting review text, combining datasets, and cleaning the data.

In [4]:
# Step 3: Data Preparation
# This step prepares the dataset for model training.
# It includes uploading files, extracting review text, combining datasets, and cleaning the data.

# Upload review files from local system into Google Colab
from google.colab import files
uploaded = files.upload()  # Opens file picker to upload all 8 review files

# Import required libraries
import os            # For file handling
import re            # For text processing using regular expressions
import pandas as pd  # For data manipulation

# Function to extract only the review text from .review files
def extract_reviews(file_path):
    """
    Reads a .review file and extracts the text inside <review_text> tags.
    This ensures that only meaningful review content is used for analysis.
    """
    with open(file_path, "r", encoding="latin-1") as file:
        content = file.read()

    # Extract text between <review_text> and </review_text>
    reviews = re.findall(r"<review_text>(.*?)</review_text>", content, re.DOTALL)
    return reviews

# List of all positive review files across domains
positive_files = [
    "books_positive.review",
    "dvd_positive.review",
    "electronics_positive.review",
    "kitchen_&_housewares_positive.review"
]

# List of all negative review files across domains
negative_files = [
    "books_negative.review",
    "dvd_negative.review",
    "electronics_negative.review",
    "kitchen_&_housewares_negative.review"
]

# Store extracted reviews
positive_reviews = []
negative_reviews = []

# Extract all positive reviews
for file in positive_files:
    positive_reviews.extend(extract_reviews(file))

# Extract all negative reviews
for file in negative_files:
    negative_reviews.extend(extract_reviews(file))

# Create DataFrame for positive reviews (label = 1)
df_pos = pd.DataFrame({
    "review": positive_reviews,
    "label": 1
})

# Create DataFrame for negative reviews (label = 0)
df_neg = pd.DataFrame({
    "review": negative_reviews,
    "label": 0
})

# Combine both datasets into one
df = pd.concat([df_pos, df_neg], ignore_index=True)

# Text cleaning function
def clean_review(text):
    """
    Cleans raw review text by:
    - Converting to lowercase
    - Removing special characters and numbers
    - Removing extra spaces
    """
    text = text.lower()  # Convert text to lowercase
    text = re.sub(r"[^a-zA-Z\s]", " ", text)  # Remove punctuation and numbers
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

# Apply cleaning function to all reviews
df["cleaned_review"] = df["review"].apply(clean_review)

# Shuffle dataset to avoid any ordering bias
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Display first few rows to verify dataset
df.head()

Saving books_negative.review to books_negative (1).review
Saving books_positive.review to books_positive (2).review
Saving dvd_negative.review to dvd_negative (2).review
Saving dvd_positive.review to dvd_positive (2).review
Saving electronics_negative.review to electronics_negative (2).review
Saving electronics_positive.review to electronics_positive (2).review
Saving kitchen_&_housewares_negative.review to kitchen_&_housewares_negative (2).review
Saving kitchen_&_housewares_positive.review to kitchen_&_housewares_positive (2).review


,review,label,cleaned_review
0,"\nBefore i bought this product, i was contempl...",1,before i bought this product i was contemplati...
1,"\nI bought this for my Tivo, hooked it up the ...",1,i bought this for my tivo hooked it up the day...
2,\nThis show really is a love it or hate it kin...,1,this show really is a love it or hate it kinda...
3,"\nI purchased this machine for my father, who ...",1,i purchased this machine for my father who is ...
4,"\nMy first Grisham novel, and it reminded me o...",0,my first grisham novel and it reminded me of w...


## 4. Initial Model: Single Domain (Books)

A baseline model was first developed using only the books domain. This helped evaluate the initial model performance before improving the dataset by combining multiple domains.

In [5]:
# Step 4: Initial Model using Books Domain Only
# This baseline model uses only the books dataset.
# It helps compare performance against the improved multi-domain model later.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Extract books-only reviews
books_positive = extract_reviews("books_positive.review")
books_negative = extract_reviews("books_negative.review")

# Create labelled dataset for books domain
books_pos_df = pd.DataFrame({
    "review": books_positive,
    "label": 1   # Positive review
})

books_neg_df = pd.DataFrame({
    "review": books_negative,
    "label": 0   # Negative review
})

# Combine positive and negative book reviews
books_df = pd.concat([books_pos_df, books_neg_df], ignore_index=True)

# Clean book reviews using the same cleaning function
books_df["cleaned_review"] = books_df["review"].apply(clean_review)

# Prepare input text and labels
X_books_text = books_df["cleaned_review"]
y_books = books_df["label"]

# Convert text into TF-IDF numerical features
books_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_books = books_vectorizer.fit_transform(X_books_text)

# Split books dataset into training and testing sets
X_books_train, X_books_test, y_books_train, y_books_test = train_test_split(
    X_books,
    y_books,
    test_size=0.2,
    random_state=42,
    stratify=y_books
)

# Train baseline Linear SVM model
books_model = LinearSVC()
books_model.fit(X_books_train, y_books_train)

# Predict on test data
books_pred = books_model.predict(X_books_test)

# Evaluate baseline model
books_accuracy = accuracy_score(y_books_test, books_pred)

print("Books Domain Accuracy:", books_accuracy)
print("\nClassification Report:")
print(classification_report(y_books_test, books_pred))

Books Domain Accuracy: 0.7775

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       200
           1       0.78      0.78      0.78       200

    accuracy                           0.78       400
   macro avg       0.78      0.78      0.78       400
weighted avg       0.78      0.78      0.78       400



## 5. Improved Model: Combined Domains

To improve the baseline result, reviews from all four domains were combined: books, DVD, electronics, and kitchen & housewares. This increased the dataset size and diversity, helping the model learn broader sentiment patterns.

Further improvements were applied to enhance model performance. Stronger TF-IDF features were used by increasing the vocabulary size and including n-grams up to trigrams, allowing the model to capture phrase-level sentiment such as “not good” or “very bad quality”.

In addition, multiple classification models were explored, including Linear SVM and Logistic Regression. Hyperparameter tuning using GridSearchCV was applied to optimise the SVM model, resulting in improved accuracy and better generalisation. These enhancements aim to achieve higher performance compared to the initial baseline model.

In [7]:
# Step 5: Improved Model using Combined Domains with Optimisation
# This model improves the combined-domain approach by using stronger TF-IDF features,
# Linear SVM tuning, and Logistic Regression comparison.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare input text and labels
X_text = df["cleaned_review"]   # Cleaned reviews from all domains
y = df["label"]                 # Sentiment labels: 1 = positive, 0 = negative

# Improved TF-IDF feature extraction
# ngram_range=(1,3) captures single words, two-word phrases, and three-word phrases.
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 3),
    stop_words="english",
    min_df=2
)

X = vectorizer.fit_transform(X_text)

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model 1: Logistic Regression
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_accuracy = accuracy_score(y_test, log_pred)

print("Logistic Regression Accuracy:", log_accuracy)
print(classification_report(y_test, log_pred))

# Model 2: Tuned Linear SVM using GridSearchCV
params = {
    "C": [0.5, 1, 2, 5]
}

grid = GridSearchCV(
    LinearSVC(),
    params,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

best_svm_model = grid.best_estimator_
svm_pred = best_svm_model.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_pred)

print("Best SVM Parameter:", grid.best_params_)
print("Tuned SVM Accuracy:", svm_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, svm_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

Logistic Regression Accuracy: 0.83375
              precision    recall  f1-score   support

           0       0.83      0.84      0.84       800
           1       0.84      0.82      0.83       800

    accuracy                           0.83      1600
   macro avg       0.83      0.83      0.83      1600
weighted avg       0.83      0.83      0.83      1600

Best SVM Parameter: {'C': 0.5}
Tuned SVM Accuracy: 0.8325

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       800
           1       0.83      0.83      0.83       800

    accuracy                           0.83      1600
   macro avg       0.83      0.83      0.83      1600
weighted avg       0.83      0.83      0.83      1600

Confusion Matrix:
[[668 132]
 [136 664]]


## 6. Model Optimisation and Comparison  

To further improve model performance, additional optimisation techniques were applied. This includes comparing different classifiers and tuning model parameters to achieve better accuracy and generalisation.

Logistic Regression was used as an alternative model to evaluate performance differences, while Linear SVM was further optimised using GridSearchCV to identify the best hyperparameter settings. These improvements aim to enhance the model’s ability to classify sentiment more accurately.

### Model Performance Comparison

The table below summarises the performance improvement across different stages of the model development process.

| Model | Dataset | Accuracy |
|------|--------|--------|
| Baseline | Books | 77.75% |
| Improved | Combined Domains | 82.65% |
| Optimised | Combined + Tuned Logistic Regression | 83.38% |

This comparison clearly demonstrates that increasing dataset diversity and applying model optimisation techniques significantly improved classification performance.performance.

In [8]:
# Step 6: Model Optimisation and Comparison
# This step improves the model by:
# 1. Comparing Logistic Regression with SVM
# 2. Tuning SVM using GridSearchCV to find best parameters

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ===============================
# Model 1: Logistic Regression
# ===============================
# Logistic Regression is a strong baseline model for text classification

log_model = LogisticRegression(max_iter=2000)  # Increase iterations for better convergence
log_model.fit(X_train, y_train)

# Predict using Logistic Regression
log_pred = log_model.predict(X_test)

# Evaluate Logistic Regression
log_accuracy = accuracy_score(y_test, log_pred)

print("Logistic Regression Accuracy:", log_accuracy)
print("\nClassification Report (Logistic Regression):")
print(classification_report(y_test, log_pred))


# ===============================
# Model 2: Tuned Linear SVM
# ===============================
# GridSearchCV is used to find the best value of C (regularisation parameter)

params = {
    "C": [0.5, 1, 2, 5]   # Different values to test
}

grid = GridSearchCV(
    LinearSVC(),
    params,
    cv=5,                # 5-fold cross-validation
    scoring="accuracy"
)

# Train GridSearch model
grid.fit(X_train, y_train)

# Get best model after tuning
best_svm_model = grid.best_estimator_

# Predict using tuned SVM
svm_pred = best_svm_model.predict(X_test)

# Evaluate tuned SVM
svm_accuracy = accuracy_score(y_test, svm_pred)

print("\nBest SVM Parameter:", grid.best_params_)
print("Tuned SVM Accuracy:", svm_accuracy)

print("\nClassification Report (Tuned SVM):")
print(classification_report(y_test, svm_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

Logistic Regression Accuracy: 0.83375

Classification Report (Logistic Regression):
              precision    recall  f1-score   support

           0       0.83      0.84      0.84       800
           1       0.84      0.82      0.83       800

    accuracy                           0.83      1600
   macro avg       0.83      0.83      0.83      1600
weighted avg       0.83      0.83      0.83      1600


Best SVM Parameter: {'C': 0.5}
Tuned SVM Accuracy: 0.8325

Classification Report (Tuned SVM):
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       800
           1       0.83      0.83      0.83       800

    accuracy                           0.83      1600
   macro avg       0.83      0.83      0.83      1600
weighted avg       0.83      0.83      0.83      1600

Confusion Matrix:
[[668 132]
 [136 664]]


### Model Comparison Result

Logistic Regression achieved the highest accuracy (approximately 83.4%), while the tuned Linear SVM achieved around 83.3%. Although both models performed similarly, Logistic Regression was selected as the final model due to its slightly better performance and stability.

This comparison highlights the importance of evaluating multiple models and selecting the most effective one based on performance metrics.

## 7. Testing the Final Model with New Sentences

The best-performing model from Step 6 was Logistic Regression. This step tests the final model using new, unseen review sentences to evaluate how well it generalises to real-world inputs.

In [9]:
# Step 7: Test final model with new review sentences
# This step uses the selected final model: Logistic Regression.
# New sentences are used to check whether the model can generalise to unseen data.

def predict_sentiment(review_text):
    """
    Takes a new review as input, cleans it using the same cleaning function,
    converts it into TF-IDF features, and predicts whether the review is
    positive or negative using the final Logistic Regression model.
    """

    # Clean the new input review using the same preprocessing function
    cleaned = clean_review(review_text)

    # Convert cleaned text into TF-IDF numerical features
    vector = vectorizer.transform([cleaned])

    # Predict sentiment using final Logistic Regression model
    prediction = log_model.predict(vector)

    # Return readable output
    return "Positive Review" if prediction[0] == 1 else "Negative Review"


# Test model with new unseen review sentences
print(predict_sentiment("This product is amazing and works perfectly"))
print(predict_sentiment("Very bad quality and waste of money"))
print(predict_sentiment("It is okay, not great but not terrible"))

Positive Review
Negative Review
Negative Review


### Testing Interpretation

The model correctly classified clearly positive and negative reviews, demonstrating strong performance on distinct sentiment patterns.

However, the sentence “It is okay, not great but not terrible” was classified as negative, indicating that the model struggles with neutral or mixed sentiment. This highlights a limitation of traditional machine learning models, which rely on keyword patterns rather than full contextual understanding.

## 8. Final Model Evaluation and Discussion  

This section summarises the performance of the final selected model (Logistic Regression) and provides interpretation of the results. It highlights the strengths, limitations, and potential improvements of the model in real-world sentiment analysis tasks.

In [10]:
# Step 8: Final Model Evaluation and Discussion
# This step presents the final model accuracy and key observations.
# It also reflects on model performance and limitations.

# Print final accuracy of selected model (Logistic Regression)
print("Final Model Accuracy:", log_accuracy)

# Interpretation of results
print("\nModel Evaluation:")

# Strengths
print("- The model performs well on clearly positive and negative reviews")
print("- Balanced precision and recall indicate consistent performance across both classes")
print("- Logistic Regression achieved the highest accuracy among tested models")

# Improvements applied
print("- Combining multiple domains improved generalisation")
print("- TF-IDF with n-grams helped capture phrase-level sentiment patterns")
print("- Model comparison and tuning improved performance")

# Limitations
print("- The model struggles with neutral or mixed sentiment reviews")
print("- It relies on keyword patterns rather than deep contextual understanding")

# Future improvements
print("\nFuture Improvements:")
print("- Use larger datasets for better generalisation")
print("- Apply advanced models such as LSTM or BERT for deeper context understanding")
print("- Further optimise hyperparameters and feature engineering")

Final Model Accuracy: 0.83375

Model Evaluation:
- The model performs well on clearly positive and negative reviews
- Balanced precision and recall indicate consistent performance across both classes
- Logistic Regression achieved the highest accuracy among tested models
- Combining multiple domains improved generalisation
- TF-IDF with n-grams helped capture phrase-level sentiment patterns
- Model comparison and tuning improved performance
- The model struggles with neutral or mixed sentiment reviews
- It relies on keyword patterns rather than deep contextual understanding

Future Improvements:
- Use larger datasets for better generalisation
- Apply advanced models such as LSTM or BERT for deeper context understanding
- Further optimise hyperparameters and feature engineering


### Key Findings

- The baseline model using only the books dataset achieved moderate accuracy (~77%), indicating limited generalisation due to domain-specific data.

- Combining multiple domains significantly improved model performance (~82%), demonstrating the importance of dataset diversity in sentiment analysis.

- Enhanced TF-IDF feature engineering with n-grams improved the model’s ability to capture phrase-level sentiment patterns.

- Logistic Regression slightly outperformed the tuned Linear SVM, achieving the highest accuracy (~83.4%) and was selected as the final model.

- The model performs well on clearly positive and negative reviews but struggles with neutral or mixed sentiment.

- Model performance is influenced by dataset size and feature representation, highlighting the importance of data quality and preprocessing.

- Further improvements would require larger datasets and advanced models (e.g., BERT) for better contextual understanding.

## Conclusion

This project developed and evaluated a sentiment analysis model using the Amazon review dataset. The initial model trained on a single domain (books) achieved moderate accuracy (approximately 77%), highlighting limitations in generalisation due to restricted data diversity. To address this, reviews from multiple domains were combined, resulting in improved performance (approximately 82%). Further optimisation through enhanced TF-IDF feature engineering, inclusion of n-grams, and model comparison led to the selection of Logistic Regression as the final model, achieving the highest accuracy (approximately 83.4%).

The results demonstrate that increasing dataset diversity and applying appropriate feature extraction and model tuning techniques significantly improve classification performance. While the model performs well on clearly positive and negative reviews, it shows limitations in handling neutral or contextually complex sentiment. Overall, the project highlights the effectiveness of traditional machine learning approaches for sentiment analysis, while also indicating that further improvements could be achieved through larger datasets and advanced models such as BERT for deeper contextual understanding.